In [2]:
!pip install -q openai

In [31]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get('GROQ_API_KEY'),
    base_url="https://api.groq.com/openai/v1"
)
MODEL = "openai/gpt-oss-20b"
print("API client setup completed!")

API client setup completed!


In [32]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ]
)
print(response.choices[0].message.content)

Hello, I hope you are having a wonderful day!


In [8]:
messages = [
    {
        "role": "system",
        "content": "You are a friendly assistant. Give short answers."
    },
    {
        "role": "user",
        "content": "Hi, my name is Lasya."
    },
    {
        "role": "assistant",
        "content": "Hello Lasya! Nice to meet you."
    },
    {
        "role": "user",
        "content": "What is my name?"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages
)

print(response.choices[0].message.content)

Your name is Lasya.


In [11]:
SYSTEM = """
ROLE:
You are a friendly assistant for Sweet Crumbs Bakery.

FACTS:
We sell:
- Plum Cake: Rs. 500
- Chocolate Cake: Rs. 650
- Cupcake: Rs. 80

# RULES:
# - Answer politely.
# - Only provide information about our bakery.
# - Do not make up prices.

FORMAT:
Keep answers short and simple.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "What cakes do you sell?"}
    ]
)

print(response.choices[0].message.content)

We sell:

- Plum Cake  
- Chocolate Cake  
- Cupcake


In [12]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Can you deliver a cake in 1 hour?"}
    ]
)

print(response.choices[0].message.content)

I’m not sure if we can deliver in 1 hour.


In [13]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": """
            Classify each customer message into one of these categories:
            ORDER, ENQUIRY, or COMPLAINT.

            Examples:
            Message: I want to buy two cupcakes.
            Category: ORDER

            Message: What is the price of chocolate cake?
            Category: ENQUIRY

            Message: My cake arrived damaged.
            Category: COMPLAINT

            Return only the category.
            """
        },
        {
            "role": "user",
            "content": "Can I order one plum cake?"
        }
    ]
)

print(response.choices[0].message.content)

ORDER


In [14]:
import json

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": """
            Extract the order details from the customer's message.
            Return only valid JSON with these fields:
            name, item, qty, eggless.
            """
        },
        {
            "role": "user",
            "content": "My name is Ravi. I want 2 plum cakes, both eggless."
        }
    ]
)

result = response.choices[0].message.content

print(result)

{"name":"Ravi","item":"plum cake","qty":2,"eggless":true}


In [15]:
import json

try:
    order = json.loads(result)

    print("Customer name:", order["name"])
    print("Item:", order["item"])
    print("Quantity:", order["qty"])
    print("Eggless:", order["eggless"])

except json.JSONDecodeError:
    print("Invalid JSON response!")

Customer name: Ravi
Item: plum cake
Quantity: 2
Eggless: True


In [16]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

total = MENU[order["item"]] * order["qty"]

print("Item:", order["item"])
print("Quantity:", order["qty"])
print("Total price: Rs.", total)

Item: plum cake
Quantity: 2
Total price: Rs. 1000


In [17]:
history = [
    {"role": "system", "content": "You are a friendly assistant."}
]

history.append({
    "role": "user",
    "content": "My name is Lasya."
})

response = client.chat.completions.create(
    model=MODEL,
    messages=history
)

reply = response.choices[0].message.content

history.append({
    "role": "assistant",
    "content": reply
})

print(reply)

Nice to meet you, Lasya! How can I help you today?


In [18]:
history.append({
    "role": "user",
    "content": "What is my name?"
})

response = client.chat.completions.create(
    model=MODEL,
    messages=history
)

reply = response.choices[0].message.content

history.append({
    "role": "assistant",
    "content": reply
})

print(reply)

Your name is Lasya.


In [19]:
MAX_TURNS = 2

recent_history = [history[0]] + history[1:][-MAX_TURNS:]

print(recent_history)

[{'role': 'system', 'content': 'You are a friendly assistant.'}, {'role': 'user', 'content': 'What is my name?'}, {'role': 'assistant', 'content': 'Your name is Lasya.'}]


In [20]:
response = client.chat.completions.create(
    model=MODEL,
    messages=recent_history + [
        {"role": "user", "content": "What is my name?"}
    ]
)

print(response.choices[0].message.content)

I’m not sure what you’d like me to call you. Could you let me know your name?


In [21]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

def get_price(item: str, qty: int = 1) -> int:
    """Return the total price for the given bakery item and quantity."""
    return MENU[item.lower().strip()] * qty

print(get_price("plum cake", 2))
print(get_price("cupcake", 3))

1000
240


In [22]:
def my_decorator(func):
    def wrapper():
        print("Before function execution")
        func()
        print("After function execution")
    return wrapper

@my_decorator
def greet():
    print("Hello, Lasya!")

greet()

Before function execution
Hello, Lasya!
After function execution


In [23]:
from openai import RateLimitError, APIConnectionError, AuthenticationError

try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": "Hello!"}
        ]
    )

    print(response.choices[0].message.content)

except RateLimitError:
    print("Too many requests. Please try again later.")

except AuthenticationError:
    print("Invalid API key. Please check your key.")

except APIConnectionError:
    print("Connection problem. Please check your internet.")

except Exception as e:
    print("Something went wrong:", e)

Hello! 👋 How can I help you today?


In [24]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

SYSTEM = """
You are a friendly customer support assistant for Sweet Crumbs Bakery.

MENU:
- Plum Cake: Rs. 500
- Chocolate Cake: Rs. 650
- Cupcake: Rs. 80

RULES:
- Answer politely.
- Only provide bakery-related information.
- Never invent prices.
- If you don't know something, say so.
- Keep answers short and simple.
"""

history = [
    {"role": "system", "content": SYSTEM}
]

MAX_TURNS = 8

print("Sweet Crumbs AI Chatbot is ready!")

Sweet Crumbs AI Chatbot is ready!


In [25]:
from openai import RateLimitError, APIConnectionError, AuthenticationError

def chat(user_text: str) -> str:
    """Send a customer message to the bakery AI and return its reply."""

    history.append({
        "role": "user",
        "content": user_text
    })

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."

    except AuthenticationError:
        reply = "Authentication failed. Please check the API key."

    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again."

    except Exception as e:
        reply = f"Something went wrong: {e}"

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

print("Chat function created successfully!")

Chat function created successfully!


In [26]:
print(chat("Hi! What items are available in your bakery?"))

Hello! We offer:

- Plum Cake – Rs. 500  
- Chocolate Cake – Rs. 650  
- Cupcake – Rs. 80  

Let me know if you’d like to order or need more details!


In [27]:
print(chat("What is the price of the chocolate cake?"))

The Chocolate Cake costs Rs. 650.


In [28]:
print(chat("Which bakery item did I ask about just now?"))

You just asked about the Chocolate Cake.


In [29]:
print(chat("My favourite item is cupcake."))
print(chat("What is your bakery timing?"))
print(chat("Do you offer chocolate cake?"))

That’s a tasty choice! A Cupcake is Rs. 80.
I’m sorry, but I don’t have the bakery’s opening hours.
Yes, we offer the Chocolate Cake. It’s Rs. 650.


In [30]:
print(chat("What is my favourite item?"))

Your favourite item is a Cupcake.
